In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
"""
SG-TAN: Sleep-Graph Temporal Abstraction Network — v3 (GPU + CPU Safe)
=======================================================================
Fixes vs v2:
  [G1] CUDNN OOM: 256Hz STFT → 64Hz (4x reduction, ≤30Hz content preserved)
       F_concat at 64Hz = 33+65+129 = 227  (vs 899 at 256Hz → CUDNN crash)
  [G2] GPU memory growth + 12GB cap to reserve CUDNN workspace headroom
  [G3] Two-phase training: encoder warmup (lazy) → embedding cache → graph-only
       Peak RAM: 2 subjects max, never all 21 simultaneously
  [G4] Sparse GATv2: O(E×d) edge-list attention, not O(N²×d) dense matrix
  [G5] Shape verification function runs before any training

Architecture:
  Stage 1: MultiResSTFT(64Hz) + ChannelAttention + Conv2D → z ∈ R^64 per epoch
  Stage 2: Build night graph (temporal k-NN + stage-transition edges)
           Sparse GATv2 (3 layers) + HierarchicalReadout → P(Insomnia)

Validation: Strict Leave-One-Subject-Out (LOSO), N=22 subjects
"""

import os
import gc
import time
import numpy as np
import tensorflow as tf
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field

# ============================================================
# 0. REPRODUCIBILITY
# ============================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ============================================================
# 1. HARDWARE CONFIG (GPU memory cap — prevents CUDNN OOM)
# ============================================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        # Reserve 1.7GB for CUDNN workspace. Without this cap,
        # CUDNN's algorithm search exhausts all 13.7GB on T4.
        tf.config.experimental.set_virtual_device_configuration(
            gpus[0],
            [tf.config.experimental.VirtualDeviceConfiguration(
                memory_limit=12288  # 12GB cap, 1.7GB CUDNN headroom
            )]
        )
        print(f"[HARDWARE] GPU: {gpus[0].name} | Cap: 12288 MB")
    except RuntimeError as e:
        # Memory growth must be set before GPUs are initialized
        print(f"[HARDWARE] GPU config warning: {e}")
else:
    print("[HARDWARE] CPU-only. Sparse attention + lazy loading active.")


# ============================================================
# 2. CONFIG — Single source of truth for all hyperparameters
# ============================================================
@dataclass
class Config:
    # Signal processing
    orig_sfreq:      int   = 256
    # TARGET IS 64Hz, NOT 256Hz.
    # Reason: 1-40Hz bandpass was applied during preprocessing.
    # Nyquist for 40Hz = 80Hz. 64Hz > 80Hz → no aliasing.
    # 4x decimation reduces epoch from 7680 → 1920 samples.
    # STFT F_concat: 33+65+129=227 (vs 899 at 256Hz → CUDNN OOM).
    target_sfreq:    int   = 64
    epoch_samples:   int   = 1920      # 30s × 64Hz
    n_channels:      int   = 9

    # Encoder
    embed_dim:       int   = 64
    enc_batch_sz:    int   = 16        # Epochs per mini-batch in encoder

    # Graph
    k_temporal:      int   = 5
    gat_dim:         int   = 64
    gat_heads:       int   = 4
    gat_layers:      int   = 3
    gat_dropout:     float = 0.1
    n_stages:        int   = 5

    # Classifier
    clf_dropout:     float = 0.3

    # Two-phase training
    lr_warmup:       float = 3e-4     # Phase 1: all parameters
    lr_graph:        float = 1e-4     # Phase 2: GATv2 + classifier only
    n_warmup_epochs: int   = 5        # Encoder warmup epochs (lazy loading)
    n_graph_epochs:  int   = 25       # Graph-only epochs (cached embeddings)
    patience:        int   = 8

    # Paths
    data_dir:        str   = "/kaggle/input/datasets/johanliebert28/insominia-sliced-chronological-dataset"

    # Debug
    dry_run:         bool  = False     # If True: verify shapes and exit

CFG = Config()


# ============================================================
# 3. DATA LOADING (lazy — one subject at a time)
# ============================================================
@dataclass
class SubjectData:
    """
    Lightweight container. Raw arrays only — no TF tensors.
    Normalization happens in prepare_graph_inputs(), not here.
    """
    subject_id:   str
    class_label:  int            # 0=Normal, 1=Insomnia
    epochs:       np.ndarray     # (N, 9, 1920) float32 at 64Hz
    stage_labels: np.ndarray     # (N,) int32   0=Wake,1=N1,2=N2,3=N3,4=REM
    times:        np.ndarray     # (N,) float64 elapsed seconds (chronological)
    n_epochs:     int = field(init=False)

    def __post_init__(self):
        self.n_epochs = self.epochs.shape[0]
        assert self.epochs.shape[1:] == (CFG.n_channels, CFG.epoch_samples), \
            (f"{self.subject_id}: expected (N,{CFG.n_channels},"
             f"{CFG.epoch_samples}), got {self.epochs.shape}")
        assert len(self.stage_labels) == self.n_epochs
        assert len(self.times) == self.n_epochs
        nan_count = np.isnan(self.epochs).sum()
        assert nan_count == 0, f"{self.subject_id}: {nan_count} NaN values"


def load_subject(data_dir: str, subject_id: str) -> SubjectData:
    """
    Load one subject. Downsample 256Hz → 64Hz immediately.
    Releases raw 256Hz array before returning.
    """
    raw = np.load(
        os.path.join(data_dir, f"{subject_id}_data.npy")
    ).astype(np.float32)                        # (N, 9, 7680) at 256Hz

    factor      = CFG.orig_sfreq // CFG.target_sfreq   # = 4
    downsampled = raw[:, :, ::factor].copy()            # (N, 9, 1920)

    del raw
    gc.collect()

    labels = np.load(
        os.path.join(data_dir, f"{subject_id}_labels.npy")
    ).astype(np.int32)

    times = np.load(
        os.path.join(data_dir, f"{subject_id}_times.npy")
    ).astype(np.float64)

    return SubjectData(
        subject_id=subject_id,
        class_label=0 if subject_id.startswith('Normal') else 1,
        epochs=downsampled,
        stage_labels=labels,
        times=times
    )


def get_all_subject_ids(data_dir: str) -> List[str]:
    """Return sorted subject IDs. Raises on duplicates."""
    ids, seen = [], set()
    for fname in sorted(os.listdir(data_dir)):
        if fname.endswith('_data.npy'):
            sid = fname.replace('_data.npy', '')
            if sid in seen:
                raise ValueError(f"DUPLICATE SUBJECT: {sid} — data leakage risk.")
            seen.add(sid)
            ids.append(sid)
    return ids


def normalize_subject_epochs(epochs: np.ndarray) -> np.ndarray:
    """
    Z-score per channel. Statistics from full night (no epoch-level leakage).
    Input/Output: (N, 9, 1920) float32
    """
    n_ep, n_ch, n_samp = epochs.shape
    flat  = epochs.transpose(1, 0, 2).reshape(n_ch, -1)   # (9, N*1920)
    mu    = flat.mean(axis=1, keepdims=True)
    sigma = flat.std(axis=1, keepdims=True)
    sigma = np.where(sigma < 1e-8, 1.0, sigma)
    normed = ((flat - mu) / sigma).reshape(n_ch, n_ep, n_samp).transpose(1, 0, 2)
    return normed.astype(np.float32)


# ============================================================
# 4. GRAPH CONSTRUCTION (THE CORE NOVELTY)
# ============================================================
def build_night_graph(
    stage_labels: np.ndarray,
    times: np.ndarray,
    k_temporal: int = 5,
    tau: float = 120.0,
    add_transition_edges: bool = True
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Construct sparse edge list with Exponential Time-Decay weights.
    """
    n = len(times)
    edge_set: dict = {}  # (i,j) -> weight (i<j)

    # Type 1: Temporal k-NN with Exponential Decay
    for i in range(n):
        start = max(0, i - k_temporal)
        end = min(n, i + k_temporal + 1)
        
        for j in range(start, end):
            if i == j: continue
            
            # Absolute temporal distance
            delta_t = abs(times[i] - times[j])
            
            # Exponential decay weight
            weight = np.exp(-max(0, delta_t - 30.0) / tau)
            if weight > 0.01:
                key = (min(i, j), max(i, j))
                edge_set[key] = max(edge_set.get(key, 0.0), weight)

    # Type 2: Stage-transition edges (Fragmentation)
    if add_transition_edges:
        for i in range(n - 1):
            if stage_labels[i] != stage_labels[i + 1]:
                # Only boost if they are actually adjacent in time (<= 30s)
                if abs(times[i] - times[i+1]) <= 30.0:
                    key = (i, i + 1)
                    # Double the weight, max capped at 2.0
                    edge_set[key] = min(edge_set.get(key, 0.0) * 2.0, 2.0)

    if not edge_set:
        # Fallback for degenerate graphs
        return np.array([[0, 0]], dtype=np.int32), np.array([1.0], dtype=np.float32)

    # Convert to PyG/TF Sparse format
    edge_keys = np.array(list(edge_set.keys()), dtype=np.int32)        # (E, 2)
    edge_weights = np.array(list(edge_set.values()), dtype=np.float32) # (E,)

    # GATv2 requires explicit bidirectional edges in the list
    edge_index = np.concatenate([edge_keys, edge_keys[:, ::-1]], axis=0)
    edge_weight = np.concatenate([edge_weights, edge_weights], axis=0)

    return edge_index, edge_weight


# ============================================================
# 5. EPOCH ENCODER
# ============================================================
class MultiResSTFT(tf.keras.layers.Layer):
    """
    Multi-resolution STFT tokenizer at 64Hz.

    Window sizes at 64Hz:
      64  samples = 1s  → captures spindles (12-15Hz), K-complexes
      128 samples = 2s  → standard bands: delta, theta, alpha, beta
      256 samples = 4s  → slow oscillations (<1Hz), high-res delta

    F dimensions: (64//2+1)=33, (128//2+1)=65, (256//2+1)=129
    F_concat = 33+65+129 = 227 (vs 899 at 256Hz → CUDNN OOM prevented)

    Input:  (batch, 9, 1920)
    Output: (batch, 9, target_t, 227)
    """

    def __init__(self, target_t: int = 20, **kwargs):
        super().__init__(**kwargs)
        self.stft_params = [
            (64,   32,  64),    # 1s window, 0.5s hop
            (128,  64, 128),    # 2s window, 1s hop
            (256, 128, 256),    # 4s window, 2s hop
        ]
        self.target_t = target_t

    def call(self, x):
        """x: (batch, 9, 1920) → (batch, 9, target_t, 227)"""
        aligned = []
        for frame_len, frame_step, fft_len in self.stft_params:
            mag = tf.math.log1p(
                tf.abs(tf.signal.stft(
                    x,
                    frame_length=frame_len,
                    frame_step=frame_step,
                    fft_length=fft_len
                ))
            )  # (batch, 9, T, F)

            shape  = tf.shape(mag)
            b, ch, t, f = shape[0], shape[1], shape[2], shape[3]

            # Align time axis to target_t via bilinear resize
            flat = tf.reshape(mag, [-1, t, f, 1])           # (B*9, T, F, 1)
            flat = tf.image.resize(
                flat, [self.target_t, f], method='bilinear'
            )                                                # (B*9, target_t, F, 1)
            flat = tf.reshape(flat, [b, ch, self.target_t, f])
            aligned.append(flat)

        return tf.concat(aligned, axis=-1)  # (batch, 9, target_t, 227)


class ChannelAttention(tf.keras.layers.Layer):
    """
    Squeeze-and-Excitation channel attention.
    Learns which EEG/EMG channels are discriminative.
    Interpretable: weights should highlight central EEG for spindles,
    occipital for alpha, EMG for REM atonia.

    Input/Output: (batch, 9, T, F)
    """

    def __init__(self, n_channels: int = 9, reduction: int = 3, **kwargs):
        super().__init__(**kwargs)
        self.gap = tf.keras.layers.GlobalAveragePooling2D()
        self.fc1 = tf.keras.layers.Dense(max(1, n_channels // reduction),
                                         activation='relu')
        self.fc2 = tf.keras.layers.Dense(n_channels, activation='sigmoid')

    def call(self, x):
        # (batch, 9, T, F) → permute → (batch, T, F, 9) for GAP
        x_perm = tf.transpose(x, [0, 2, 3, 1])
        attn   = self.fc2(self.fc1(self.gap(x_perm)))    # (batch, 9)
        attn   = tf.reshape(attn, [-1, tf.shape(x)[1], 1, 1])
        return x * attn


class EpochEncoder(tf.keras.Model):
    """
    Stage 1: (batch, 9, 1920) → (batch, embed_dim=64)

    Peak GPU memory per batch of 16 epochs at 64Hz:
      STFT output: 16 × 9 × 20 × 227 × 4B ≈ 26MB
      Conv2D activations: ~30MB
      Total: ~56MB (vs ~450MB at 256Hz with batch=32)
    """

    def __init__(self, embed_dim: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.stft    = MultiResSTFT(target_t=20)
        self.ch_attn = ChannelAttention(n_channels=9, reduction=3)

        self.conv_block = tf.keras.Sequential([
            tf.keras.layers.Conv2D(16,  (3, 3), padding='same', activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPool2D((2, 2)),
            tf.keras.layers.Conv2D(32,  (3, 3), padding='same', activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPool2D((2, 2)),
            tf.keras.layers.Conv2D(64,  (3, 3), padding='same', activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.GlobalAveragePooling2D(),
        ])

        self.proj = tf.keras.layers.Dense(embed_dim)
        self.ln   = tf.keras.layers.LayerNormalization()

    def call(self, x, training=False):
        """x: (batch, 9, 1920) → (batch, embed_dim)"""
        spec = self.stft(x)                        # (batch, 9, 20, 227)
        spec = self.ch_attn(spec)                  # (batch, 9, 20, 227)
        spec = tf.transpose(spec, [0, 2, 3, 1])    # (batch, 20, 227, 9)
        h    = self.conv_block(spec, training=training)
        return self.ln(self.proj(h))               # (batch, embed_dim)

    def encode_batched(
        self,
        epochs_np:  np.ndarray,
        batch_size: int  = 16,
        training:   bool = False
    ) -> np.ndarray:
        """
        Encode full subject (N epochs) in mini-batches.
        Releases each batch from GPU/CPU memory after accumulation.

        Input:  (N, 9, 1920) numpy
        Output: (N, embed_dim) numpy
        """
        parts = []
        for start in range(0, epochs_np.shape[0], batch_size):
            end   = min(start + batch_size, epochs_np.shape[0])
            batch = tf.constant(epochs_np[start:end], dtype=tf.float32)
            z     = self(batch, training=training)
            parts.append(z.numpy())
            del batch, z
        return np.concatenate(parts, axis=0)


# ============================================================
# 6. SPARSE GATv2 LAYER
# ============================================================
class SparseGATv2Layer(tf.keras.layers.Layer):
    """
    Graph Attention Network v2 (Brody et al., ICLR 2022) — Sparse implementation.

    Memory comparison for N=780 epochs, d=64:
      Dense (v2): N×N×d = 780×780×64×4B = 156MB per layer × 3 = 468MB
      Sparse (v3): E×d ≈ 4000×64×4B = 1MB per layer × 3 = 3MB

    The sparse implementation is mathematically identical to dense GATv2
    but operates only on existing edges, making it viable on CPU and GPU alike.
    """

    def __init__(
        self,
        out_dim:      int,
        n_heads:      int   = 4,
        dropout_rate: float = 0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        assert out_dim % n_heads == 0, \
            f"out_dim ({out_dim}) must be divisible by n_heads ({n_heads})"
        self.n_heads      = n_heads
        self.head_dim     = out_dim // n_heads
        self.out_dim      = out_dim
        self.dropout      = tf.keras.layers.Dropout(dropout_rate)
        self.ln           = tf.keras.layers.LayerNormalization()
        self.leaky        = tf.keras.layers.LeakyReLU(negative_slope=0.2)
        self.W_l          = tf.keras.layers.Dense(out_dim, use_bias=False)
        self.W_r          = tf.keras.layers.Dense(out_dim, use_bias=False)
        self._attn_vec    = None

    def build(self, input_shape):
        """Explicit build() to prevent Keras 'unbuilt state' warnings."""
        d_in = input_shape[-1]
        self._attn_vec = self.add_weight(
            name='attn_vec',
            shape=(self.n_heads, self.head_dim),
            initializer='glorot_uniform',
            trainable=True
        )
        # Trigger Dense weight creation with correct input dim
        dummy = tf.zeros([1, d_in])
        self.W_l(dummy)
        self.W_r(dummy)
        del dummy
        super().build(input_shape)

    def call(
        self,
        node_features: tf.Tensor,   # (N, d_in)
        edge_index:    tf.Tensor,   # (E, 2) int32
        edge_weight:   tf.Tensor,   # (E,)   float32
        training:      bool = False
    ) -> Tuple[tf.Tensor, tf.Tensor]:
        """Returns: (N, out_dim), (E, n_heads)"""
        N = tf.shape(node_features)[0]
        E = tf.shape(edge_index)[0]

        h_l = self.W_l(node_features)   # (N, out_dim)
        h_r = self.W_r(node_features)   # (N, out_dim)

        src = edge_index[:, 0]           # (E,)
        dst = edge_index[:, 1]           # (E,)

        # GATv2: e_ij = a^T LeakyReLU(W_l h_i + W_r h_j) — only over edges
        pair = self.leaky(
            tf.gather(h_l, src) + tf.gather(h_r, dst)
        )  # (E, out_dim)

        pair_h = tf.reshape(pair, [E, self.n_heads, self.head_dim])
        scores = tf.reduce_sum(
            pair_h * self._attn_vec[tf.newaxis], axis=-1
        )  # (E, n_heads)

        # Scale by edge weight (transition edges get 2× attention)
        scores = scores * tf.reshape(edge_weight, [-1, 1])

        # Sparse softmax: normalize per destination node
        attn_w = self._sparse_softmax(scores, dst, N)
        attn_w = self.dropout(attn_w, training=training)  # (E, n_heads)

        # Weighted aggregation via scatter_add
        h_r_h    = tf.reshape(h_r, [N, self.n_heads, self.head_dim])
        src_feat = tf.gather(h_r_h, src)                          # (E, H, D)
        weighted = src_feat * tf.reshape(attn_w, [E, self.n_heads, 1])

        out = tf.zeros([N, self.n_heads, self.head_dim], dtype=tf.float32)
        out = tf.tensor_scatter_nd_add(
            out, tf.reshape(dst, [-1, 1]), weighted
        )
        out = tf.reshape(out, [N, self.out_dim])

        # Residual + LayerNorm
        residual = node_features if node_features.shape[-1] == self.out_dim \
                   else tf.zeros_like(out)
        return self.ln(out + residual), attn_w

    @staticmethod
    def _sparse_softmax(
        scores:  tf.Tensor,   # (E, n_heads)
        dst_idx: tf.Tensor,   # (E,) int32
        n_nodes: int
    ) -> tf.Tensor:
        """Numerically stable softmax over incoming edges per node per head."""
        node_max = tf.math.unsorted_segment_max(
            scores, dst_idx, num_segments=n_nodes
        )
        exp_s    = tf.exp(scores - tf.gather(node_max, dst_idx))
        node_sum = tf.math.unsorted_segment_sum(
            exp_s, dst_idx, num_segments=n_nodes
        )
        return exp_s / (tf.gather(node_sum, dst_idx) + 1e-8)


# ============================================================
# 7. HIERARCHICAL READOUT
# ============================================================
class HierarchicalReadout(tf.keras.layers.Layer):
    """
    Attention-weighted graph pooling.
    The (N,) output weights are the interpretability signal:
    high weight = epoch drove the classification decision.

    Input:  (N, d)
    Output: (d,) graph embedding, (N,) epoch importance weights
    """

    def __init__(self, hidden_dim: int = 32, **kwargs):
        super().__init__(**kwargs)
        self.fc = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden_dim, activation='tanh'),
            tf.keras.layers.Dense(1)
        ])

    def call(self, h):
        w = tf.nn.softmax(self.fc(h), axis=0)        # (N, 1)
        return (
            tf.reduce_sum(h * w, axis=0),             # (d,)
            tf.squeeze(w, axis=-1)                    # (N,)
        )


# ============================================================
# 8. GRAPH CLASSIFIER (Stage 2 module — trained separately in Phase 2)
# ============================================================
class GraphClassifier(tf.keras.Model):
    """
    Takes pre-computed epoch embeddings z ∈ R^(N × embed_dim).
    Decoupled from EpochEncoder to allow independent weight updates.

    In Phase 2, EpochEncoder is frozen and embeddings are cached.
    Only GraphClassifier weights update → 20× faster per epoch.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.stage_embed = tf.keras.layers.Dense(16, activation='relu')
        self.input_proj  = tf.keras.layers.Dense(CFG.gat_dim, activation='relu')

        self.gat = [
            SparseGATv2Layer(
                out_dim=CFG.gat_dim,
                n_heads=CFG.gat_heads,
                dropout_rate=CFG.gat_dropout,
                name=f'gat_layer_{i}'
            )
            for i in range(CFG.gat_layers)
        ]

        self.readout = HierarchicalReadout(hidden_dim=32)

        self.clf = tf.keras.Sequential([
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dropout(CFG.clf_dropout),
            tf.keras.layers.Dense(16, activation='relu'),
            tf.keras.layers.Dropout(CFG.clf_dropout),
            tf.keras.layers.Dense(1, activation='sigmoid')
        ])

    def call(
        self,
        z:            tf.Tensor,   # (N, embed_dim)
        stage_onehot: tf.Tensor,   # (N, 5)
        edge_index:   tf.Tensor,   # (E, 2) int32
        edge_weight:  tf.Tensor,   # (E,)
        training:     bool = False
    ) -> Tuple[tf.Tensor, tf.Tensor, List[tf.Tensor]]:
        stage_feat = self.stage_embed(stage_onehot)              # (N, 16)
        h = self.input_proj(tf.concat([z, stage_feat], axis=-1)) # (N, gat_dim)

        attn_list = []
        for layer in self.gat:
            h, attn = layer(h, edge_index, edge_weight, training=training)
            attn_list.append(attn)

        graph_emb, readout_attn = self.readout(h)

        # shape (1,) — NEVER 0-d. Prevents BCE InvalidArgumentError.
        prob = tf.reshape(
            self.clf(tf.expand_dims(graph_emb, 0), training=training),
            (1,)
        )
        return prob, readout_attn, attn_list


# ============================================================
# 9. SHAPE VERIFICATION (run before any training)
# ============================================================
def verify_encoder_shapes() -> None:
    """
    Dry-run the encoder on 2 dummy epochs.
    Prints I/O shapes and F_concat. Exits cleanly.
    Call this before committing to a full LOSO run.
    """
    print("\n[SHAPE CHECK] Running encoder on dummy input...")
    dummy   = tf.zeros([2, CFG.n_channels, CFG.epoch_samples])
    encoder = EpochEncoder(embed_dim=CFG.embed_dim)

    # Trace shapes at each stage
    stft_out  = encoder.stft(dummy)
    print(f"  Input:        {dummy.shape}")
    print(f"  After STFT:   {stft_out.shape}  "
          f"(batch, ch={stft_out.shape[1]}, T={stft_out.shape[2]}, "
          f"F_concat={stft_out.shape[3]})")

    attn_out  = encoder.ch_attn(stft_out)
    conv_in   = tf.transpose(attn_out, [0, 2, 3, 1])
    print(f"  Conv2D input: {conv_in.shape}  "
          f"(batch, T, F, channels)")

    z = encoder(dummy, training=False)
    print(f"  Encoder output: {z.shape}  (batch, embed_dim={CFG.embed_dim})")

    # Build a tiny 3-node graph and verify GraphClassifier
    clf = GraphClassifier()
    z3  = tf.zeros([3, CFG.embed_dim])
    s3  = tf.zeros([3, CFG.n_stages])
    ei  = tf.constant([[0,1],[1,0],[1,2],[2,1]], dtype=tf.int32)
    ew  = tf.ones([4], dtype=tf.float32)
    p, ra, _ = clf(z3, s3, ei, ew, training=False)
    print(f"  GraphClassifier output: prob={p.shape}, "
          f"readout_attn={ra.shape}")

    del dummy, encoder, stft_out, attn_out, conv_in, z, clf, z3, s3, ei, ew, p, ra
    tf.keras.backend.clear_session()
    gc.collect()
    print("[SHAPE CHECK] All shapes valid.\n")


# ============================================================
# 10. TWO-PHASE TRAINING FOR ONE LOSO FOLD
# ============================================================
def train_one_fold(
    train_subject_ids: List[str],
    test_subject:      SubjectData,
    data_dir:          str
) -> Tuple[float, np.ndarray]:
    """
    Two-phase training strategy.

    PHASE 1 — Encoder Warmup (CFG.n_warmup_epochs, lazy loading):
      For each training epoch:
        For each training subject (loaded one at a time):
          Forward + backward through Encoder + GraphClassifier
          Delete subject data immediately after gradient step
      Peak RAM: 1 subject + model weights

    PHASE 2 — Graph Training (CFG.n_graph_epochs, cached embeddings):
      Run frozen encoder once per training subject → store (N, embed_dim) numpy
      Cache size: 21 × 780 × 64 × 4B ≈ 4MB
      Only GraphClassifier.trainable_variables are updated
      ~20× faster per epoch than Phase 1

    Returns:
      pred_prob:    float ∈ [0,1] — P(Insomnia) for test subject
      readout_attn: (N_test,) — epoch-level importance weights
    """
    encoder   = EpochEncoder(embed_dim=CFG.embed_dim)
    graph_clf = GraphClassifier()
    opt_full  = tf.keras.optimizers.Adam(learning_rate=CFG.lr_warmup)
    opt_graph = tf.keras.optimizers.Adam(learning_rate=CFG.lr_graph)
    bce       = tf.keras.losses.BinaryCrossentropy(from_logits=False)

    # ── PHASE 1: ENCODER WARMUP ──────────────────────────────
    print(f"  [Phase 1] Encoder warmup ({CFG.n_warmup_epochs} epochs, lazy loading)")

    for ep in range(CFG.n_warmup_epochs):
        ep_losses = []
        order     = np.random.permutation(len(train_subject_ids))

        for idx in order:
            sid   = train_subject_ids[idx]
            subj  = load_subject(data_dir, sid)
            normed = normalize_subject_epochs(subj.epochs)
            ei, ew = build_night_graph(subj.stage_labels, subj.times,
                                        CFG.k_temporal)
            stage_oh = np.eye(CFG.n_stages, dtype=np.float32)[subj.stage_labels]
            label_tf = tf.constant([float(subj.class_label)], dtype=tf.float32)

            # Encode in mini-batches inside tape for encoder gradient flow
            n_total = normed.shape[0]
            all_vars = (encoder.trainable_variables +
                        graph_clf.trainable_variables)

            with tf.GradientTape() as tape:
                parts = []
                for start in range(0, n_total, CFG.enc_batch_sz):
                    end = min(start + CFG.enc_batch_sz, n_total)
                    z_b = encoder(
                        tf.constant(normed[start:end], dtype=tf.float32),
                        training=True
                    )
                    parts.append(z_b)

                z        = tf.concat(parts, axis=0)
                prob, _, _ = graph_clf(
                    z,
                    tf.constant(stage_oh, dtype=tf.float32),
                    tf.constant(ei,       dtype=tf.int32),
                    tf.constant(ew,       dtype=tf.float32),
                    training=True
                )
                loss = bce(label_tf, prob)
                l2   = tf.add_n([
                    tf.nn.l2_loss(v) for v in all_vars
                    if 'bias' not in v.name
                ]) * 1e-4
                total_loss = loss + l2

            grads, _ = tf.clip_by_global_norm(
                tape.gradient(total_loss, all_vars), 1.0
            )
            opt_full.apply_gradients(zip(grads, all_vars))
            ep_losses.append(float(loss))

            del subj, normed, ei, ew, stage_oh, label_tf
            del parts, z, prob, tape, grads
            gc.collect()

        print(f"    Warmup {ep+1}/{CFG.n_warmup_epochs} "
              f"| Loss: {np.mean(ep_losses):.4f}")

    # ── PHASE 2: CACHE EMBEDDINGS → GRAPH-ONLY TRAINING ──────
    print(f"  [Phase 2] Caching embeddings → graph-only training "
          f"({CFG.n_graph_epochs} epochs)")

    cache: Dict[str, object] = {}
    for sid in train_subject_ids:
        subj  = load_subject(data_dir, sid)
        normed = normalize_subject_epochs(subj.epochs)
        z_np  = encoder.encode_batched(
            normed, batch_size=CFG.enc_batch_sz, training=False
        )
        ei, ew = build_night_graph(subj.stage_labels, subj.times, CFG.k_temporal)

        cache[sid] = {
            'z':     z_np,
            'stage': np.eye(CFG.n_stages, dtype=np.float32)[subj.stage_labels],
            'ei':    ei,
            'ew':    ew,
            'label': float(subj.class_label)
        }
        del subj, normed
        gc.collect()

    cache_mb = sum(
        v['z'].nbytes + v['stage'].nbytes + v['ei'].nbytes + v['ew'].nbytes
        for v in cache.values()
    ) / 1e6
    print(f"    Embedding cache: {cache_mb:.1f} MB")

    best_loss = float('inf')
    wait      = 0

    for ep in range(CFG.n_graph_epochs):
        ep_losses = []
        order     = np.random.permutation(len(train_subject_ids))

        for idx in order:
            sid = train_subject_ids[idx]
            c   = cache[sid]

            z_tf  = tf.constant(c['z'],     dtype=tf.float32)
            st_tf = tf.constant(c['stage'], dtype=tf.float32)
            ei_tf = tf.constant(c['ei'],    dtype=tf.int32)
            ew_tf = tf.constant(c['ew'],    dtype=tf.float32)
            lb_tf = tf.constant([c['label']],dtype=tf.float32)

            with tf.GradientTape() as tape:
                prob, _, _ = graph_clf(z_tf, st_tf, ei_tf, ew_tf, training=True)
                loss = bce(lb_tf, prob)
                l2   = tf.add_n([
                    tf.nn.l2_loss(v) for v in graph_clf.trainable_variables
                    if 'bias' not in v.name
                ]) * 1e-4
                total_loss = loss + l2

            grads, _ = tf.clip_by_global_norm(
                tape.gradient(total_loss, graph_clf.trainable_variables), 1.0
            )
            opt_graph.apply_gradients(
                zip(grads, graph_clf.trainable_variables)
            )
            ep_losses.append(float(loss))

            del z_tf, st_tf, ei_tf, ew_tf, lb_tf, tape, grads
            gc.collect()

        mean_loss = float(np.mean(ep_losses))
        if ep % 5 == 0:
            print(f"    Graph epoch {ep+1:3d}/{CFG.n_graph_epochs} "
                  f"| Loss: {mean_loss:.4f}")

        if mean_loss < best_loss - 1e-4:
            best_loss = mean_loss
            wait      = 0
        else:
            wait += 1
            if wait >= CFG.patience:
                print(f"    Early stop at graph epoch {ep+1} "
                      f"(best: {best_loss:.4f})")
                break

    # ── TEST ─────────────────────────────────────────────────
    normed_test = normalize_subject_epochs(test_subject.epochs)
    z_test      = encoder.encode_batched(
        normed_test, batch_size=CFG.enc_batch_sz, training=False
    )
    ei_test, ew_test = build_night_graph(
        test_subject.stage_labels, test_subject.times, CFG.k_temporal
    )
    stage_test = np.eye(CFG.n_stages, dtype=np.float32)[test_subject.stage_labels]

    t0 = time.perf_counter()
    prob_tf, readout_attn, _ = graph_clf(
        tf.constant(z_test,      dtype=tf.float32),
        tf.constant(stage_test,  dtype=tf.float32),
        tf.constant(ei_test,     dtype=tf.int32),
        tf.constant(ew_test,     dtype=tf.float32),
        training=False
    )
    latency_ms = (time.perf_counter() - t0) * 1000.0
    print(f"    Test inference latency: {latency_ms:.2f} ms")

    pred_prob = float(prob_tf[0])

    del encoder, graph_clf, cache
    del normed_test, z_test, ei_test, ew_test, stage_test
    tf.keras.backend.clear_session()
    gc.collect()

    return pred_prob, readout_attn.numpy()


# ============================================================
# 11. LOSO CROSS-VALIDATION
# ============================================================
def run_loso_cv(data_dir: str, subject_ids: List[str]) -> Dict:
    """
    Strict Leave-One-Subject-Out cross-validation.

    At no point are two subjects' raw data in memory simultaneously.
    The test subject is loaded once at fold start, never during training.
    """
    n          = len(subject_ids)
    all_preds  = []
    all_labels = []
    all_sids   = []
    all_attns  = {}
    fold_times = []

    for fold_idx in range(n):
        test_id    = subject_ids[fold_idx]
        train_ids  = [s for i, s in enumerate(subject_ids) if i != fold_idx]
        test_subj  = load_subject(data_dir, test_id)

        print(f"\n{'='*68}")
        print(f"FOLD {fold_idx+1:2d}/{n} — Test: {test_id} "
              f"({'Insomnia' if test_subj.class_label else 'Normal'}) "
              f"| {test_subj.n_epochs} epochs")
        print(f"{'='*68}")

        t0 = time.perf_counter()
        pred_prob, readout_attn = train_one_fold(train_ids, test_subj, data_dir)
        elapsed = time.perf_counter() - t0

        pred_label = 1 if pred_prob >= 0.5 else 0
        true_label = test_subj.class_label
        status     = "✓" if pred_label == true_label else "✗"

        all_preds.append(pred_prob)
        all_labels.append(true_label)
        all_sids.append(test_id)
        all_attns[test_id] = readout_attn
        fold_times.append(elapsed)

        print(f"\n  [{status}] P(Insomnia)={pred_prob:.4f} | "
              f"Pred={'Insomnia' if pred_label else 'Normal'} | "
              f"True={'Insomnia' if true_label else 'Normal'} | "
              f"Fold time: {elapsed/60:.1f} min")

        del test_subj
        gc.collect()

    total_h = sum(fold_times) / 3600
    print(f"\n[TIMING] Mean fold: {np.mean(fold_times)/60:.1f} min | "
          f"Total: {total_h:.2f} h")

    results = compute_metrics(all_preds, all_labels, all_sids)
    results['readout_attentions'] = all_attns
    return results


# ============================================================
# 12. METRICS (publication-grade)
# ============================================================
def compute_metrics(
    preds:       List[float],
    labels:      List[int],
    subject_ids: List[str]
) -> Dict:
    pa  = np.array(preds)
    la  = np.array(labels)
    pb  = (pa >= 0.5).astype(int)

    tp = int(np.sum((pb == 1) & (la == 1)))
    tn = int(np.sum((pb == 0) & (la == 0)))
    fp = int(np.sum((pb == 1) & (la == 0)))
    fn = int(np.sum((pb == 0) & (la == 1)))
    N  = tp + tn + fp + fn

    acc  = (tp + tn) / N
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = (2*prec*sens / (prec+sens)) if (prec+sens) > 0 else 0.0
    pe   = ((tp+fp)*(tp+fn) + (tn+fn)*(tn+fp)) / N**2
    kap  = (acc - pe) / (1 - pe) if (1 - pe) > 1e-8 else 0.0

    # AUC-ROC (exact Wilcoxon, N=22)
    sl   = la[np.argsort(-pa)]
    tp_r = 0
    asum = 0
    for lbl in sl:
        if lbl == 1: tp_r += 1
        else: asum += tp_r
    n_p  = np.sum(la == 1)
    n_n  = np.sum(la == 0)
    auc  = asum / (n_p * n_n) if (n_p * n_n) > 0 else 0.0

    # Wilson 95% CI on accuracy
    z     = 1.96
    ci_c  = (acc + z**2/(2*N)) / (1 + z**2/N)
    ci_m  = (z * np.sqrt(acc*(1-acc)/N + z**2/(4*N**2))) / (1 + z**2/N)
    ci_lo = ci_c - ci_m
    ci_hi = ci_c + ci_m

    print(f"\n{'='*68}")
    print(f"LOSO RESULTS (N={N})")
    print(f"{'='*68}")
    print(f"  Accuracy:    {acc:.4f}  95%CI [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  Sensitivity: {sens:.4f}  (Insomnia recall)")
    print(f"  Specificity: {spec:.4f}  (Normal recall)")
    print(f"  Precision:   {prec:.4f}")
    print(f"  F1-Score:    {f1:.4f}")
    print(f"  AUC-ROC:     {auc:.4f}")
    print(f"  Cohen's κ:   {kap:.4f}")
    print(f"  Confusion:   TP={tp}, TN={tn}, FP={fp}, FN={fn}")
    print(f"{'='*68}")
    print(f"\n  Per-Subject:")
    for sid, p, t in zip(subject_ids, preds, labels):
        s = "✓" if (1 if p>=0.5 else 0)==t else "✗"
        print(f"    [{s}] {sid:<15}: P(Ins)={p:.4f} | "
              f"{'Insomnia' if t else 'Normal':<8}")

    return dict(
        accuracy=acc, sensitivity=sens, specificity=spec,
        precision=prec, f1=f1, auc=auc, kappa=kap,
        ci=(ci_lo, ci_hi),
        predictions=preds, labels=labels, subject_ids=subject_ids,
        confusion=dict(tp=tp, tn=tn, fp=fp, fn=fn)
    )


# ============================================================
# 13. BASELINE: HANDCRAFTED FEATURES + RBF-SVM
# ============================================================
def run_baseline_svm(data_dir: str, subject_ids: List[str]) -> Dict:
    """
    61-feature physiological baseline + RBF-SVM, LOSO.
    Uses raw 256Hz data for PSD (not downsampled) to maximize feature quality.
    Lazy loading: one subject at a time.

    Feature vector (61 features):
      36: Absolute band power (9 ch × 4 bands: δ,θ,α,β)
      18: Relative band power ratios (9 ch × 2: θ/α, δ/β)
       5: Sleep stage distribution (% Wake, N1, N2, N3, REM)
       1: Fragmentation index (stage transitions / n_epochs)
       1: Sleep efficiency (% non-Wake epochs)
    """
    from sklearn.svm import SVC
    from sklearn.preprocessing import StandardScaler

    print(f"\n{'='*68}")
    print("BASELINE: Handcrafted Features + RBF-SVM (LOSO)")
    print(f"{'='*68}")

    sfreq = float(CFG.orig_sfreq)
    freqs = np.fft.rfftfreq(7680, 1.0 / sfreq)
    bands = {
        'delta': (freqs >= 0.5) & (freqs < 4.0),
        'theta': (freqs >= 4.0) & (freqs < 8.0),
        'alpha': (freqs >= 8.0) & (freqs < 13.0),
        'beta':  (freqs >= 13.0) & (freqs < 30.0),
    }

    def extract_features(sid: str) -> np.ndarray:
        raw    = np.load(
            os.path.join(data_dir, f"{sid}_data.npy")
        ).astype(np.float32)                         # (N, 9, 7680)
        labels = np.load(
            os.path.join(data_dir, f"{sid}_labels.npy")
        ).astype(np.int32)

        fft_mag = np.abs(np.fft.rfft(raw, axis=-1)) # (N, 9, 3841)
        feats   = []

        for ch in range(9):
            # Mean spectrum across epochs: (3841,)
            ms = fft_mag[:, ch, :].mean(axis=0)
            d  = ms[bands['delta']].mean()
            t  = ms[bands['theta']].mean()
            a  = ms[bands['alpha']].mean()
            b  = ms[bands['beta']].mean()
            feats.extend([d, t, a, b,
                          t / (a + 1e-8),
                          d / (b + 1e-8)])

        n_ep = len(labels)
        for stage in range(5):
            feats.append(float(np.sum(labels == stage)) / n_ep)
        feats.append(float(np.sum(labels[:-1] != labels[1:])) / n_ep)
        feats.append(float(np.sum(labels != 0)) / n_ep)

        del raw, fft_mag
        gc.collect()
        return np.array(feats, dtype=np.float32)  # (61,)

    print("  Extracting features (lazy loading)...")
    X = np.array([extract_features(sid) for sid in subject_ids])  # (22, 61)
    y = np.array(
        [0 if sid.startswith('Normal') else 1 for sid in subject_ids],
        dtype=np.int32
    )
    print(f"  Feature matrix: {X.shape} | NaN: {np.isnan(X).sum()} | "
          f"Inf: {np.isinf(X).sum()}")

    preds, lbls, sids = [], [], []
    for i, sid in enumerate(subject_ids):
        sc      = StandardScaler()
        X_tr    = sc.fit_transform(np.delete(X, i, axis=0))
        X_te    = sc.transform(X[i:i+1])
        svm     = SVC(kernel='rbf', C=1.0, gamma='scale',
                      probability=True, random_state=SEED)
        svm.fit(X_tr, np.delete(y, i))
        preds.append(float(svm.predict_proba(X_te)[0, 1]))
        lbls.append(int(y[i]))
        sids.append(sid)

    return compute_metrics(preds, lbls, sids)


# ============================================================
# 14. MAIN
# ============================================================
if __name__ == "__main__":
    print("=" * 68)
    print("SG-TAN v3 — Two-Phase Training, GPU + CPU Safe")
    print(f"  target_sfreq : {CFG.target_sfreq} Hz  "
          f"(downsampled from {CFG.orig_sfreq} Hz — CUDNN OOM prevention)")
    print(f"  embed_dim    : {CFG.embed_dim}")
    print(f"  gat_dim      : {CFG.gat_dim}")
    print(f"  warmup epochs: {CFG.n_warmup_epochs} (all params, lazy loading)")
    print(f"  graph epochs : {CFG.n_graph_epochs} (GATv2 + clf only, cached z)")
    print("=" * 68)

    # Step 0: Always verify shapes before training
    verify_encoder_shapes()

    if CFG.dry_run:
        print("[DRY RUN] Shape check passed. Set CFG.dry_run=False to train.")
    else:
        subject_ids = get_all_subject_ids(CFG.data_dir)
        print(f"\n[DATA] {len(subject_ids)} subjects found:")
        for sid in subject_ids:
            lbl = "Insomnia" if not sid.startswith("Normal") else "Normal"
            print(f"  {sid:<15} ({lbl})")

        # Phase 1: Baseline
        print("\n\n" + "#" * 68)
        print("# BASELINE — Handcrafted Features + RBF-SVM")
        print("#" * 68)
        baseline_results = run_baseline_svm(CFG.data_dir, subject_ids)

        # Phase 2: SG-TAN
        print("\n\n" + "#" * 68)
        print("# SG-TAN — Two-Phase LOSO")
        print("#" * 68)
        sgtan_results = run_loso_cv(CFG.data_dir, subject_ids)

        # Head-to-head
        print("\n\n" + "#" * 68)
        print("# HEAD-TO-HEAD")
        print("#" * 68)
        print(f"  {'Metric':<14} {'SVM':<12} {'SG-TAN':<12} {'Δ'}")
        print(f"  {'-'*14} {'-'*12} {'-'*12} {'-'*8}")
        for m in ['accuracy','sensitivity','specificity','f1','auc','kappa']:
            b, s  = baseline_results[m], sgtan_results[m]
            d     = s - b
            arrow = "▲" if d > 0.001 else ("▼" if d < -0.001 else "=")
            print(f"  {m:<14} {b:<12.4f} {s:<12.4f} {arrow} {d:+.4f}")

        # McNemar's test
        b_bin = [(1 if p>=0.5 else 0) for p in baseline_results['predictions']]
        s_bin = [(1 if p>=0.5 else 0) for p in sgtan_results['predictions']]
        true  = baseline_results['labels']
        bc    = np.array([int(b==t) for b, t in zip(b_bin, true)])
        sc_   = np.array([int(s==t) for s, t in zip(s_bin, true)])
        n01   = int(np.sum((bc==0) & (sc_==1)))
        n10   = int(np.sum((bc==1) & (sc_==0)))
        disc  = n01 + n10
        print(f"\n  McNemar's Test (continuity-corrected):")
        print(f"    Baseline wrong & SG-TAN right: {n01}")
        print(f"    Baseline right & SG-TAN wrong: {n10}")
        if disc > 0:
            chi2 = (abs(n01 - n10) - 1)**2 / disc
            sig  = "YES (p<0.05)" if chi2 > 3.841 else "NO (p≥0.05)"
            print(f"    χ² = {chi2:.4f} | Significant: {sig}")
        else:
            print("    No discordant pairs.")

        print(f"\n{'='*68}")
        print("DONE")
        print(f"{'='*68}")

2026-03-09 19:49:52.626474: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773085792.811181      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773085792.868934      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773085793.310585      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773085793.310629      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773085793.310634      55 computation_placer.cc:177] computation placer alr

[HARDWARE] GPU: /physical_device:GPU:0 | Cap: 12288 MB
SG-TAN v3 — Two-Phase Training, GPU + CPU Safe
  target_sfreq : 64 Hz  (downsampled from 256 Hz — CUDNN OOM prevention)
  embed_dim    : 64
  gat_dim      : 64
  warmup epochs: 5 (all params, lazy loading)
  graph epochs : 25 (GATv2 + clf only, cached z)

[SHAPE CHECK] Running encoder on dummy input...


I0000 00:00:1773085816.517032      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12288 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


  Input:        (2, 9, 1920)
  After STFT:   (2, 9, 20, 227)  (batch, ch=9, T=20, F_concat=227)
  Conv2D input: (2, 20, 227, 9)  (batch, T, F, channels)


I0000 00:00:1773085818.718624      55 cuda_dnn.cc:529] Loaded cuDNN version 91002


  Encoder output: (2, 64)  (batch, embed_dim=64)
  GraphClassifier output: prob=(1,), readout_attn=(3,)
[SHAPE CHECK] All shapes valid.


[DATA] 22 subjects found:
  Insomnia_01     (Insomnia)
  Insomnia_02     (Insomnia)
  Insomnia_03     (Insomnia)
  Insomnia_04     (Insomnia)
  Insomnia_05     (Insomnia)
  Insomnia_06     (Insomnia)
  Insomnia_07     (Insomnia)
  Insomnia_08     (Insomnia)
  Insomnia_09     (Insomnia)
  Insomnia_10     (Insomnia)
  Insomnia_11     (Insomnia)
  Normal_01       (Normal)
  Normal_02       (Normal)
  Normal_03       (Normal)
  Normal_04       (Normal)
  Normal_05       (Normal)
  Normal_06       (Normal)
  Normal_07       (Normal)
  Normal_08       (Normal)
  Normal_09       (Normal)
  Normal_10       (Normal)
  Normal_11       (Normal)


####################################################################
# BASELINE — Handcrafted Features + RBF-SVM
####################################################################

BASELINE: Handcrafted Features + RB

ValueError: Inputs must be an iterable of at least one Tensor/IndexedSlices with the same dtype and shape.

In [3]:
"""
SG-TAN: Sleep-Graph Temporal Abstraction Network (Corrected & OOM-Safe)
=================================================
Subject-level Insomnia vs Normal classification from full-night PSG.

Validation: Strict Leave-One-Subject-Out (LOSO) Cross-Validation
Hardware Target: Kaggle T4 GPU
"""

import os
import gc
import time
import numpy as np
import tensorflow as tf
from typing import Dict, List, Tuple
from dataclasses import dataclass, field

# ============================================================
# 0. REPRODUCIBILITY & HARDWARE CONFIG
# ============================================================
# ============================================================
# 0. REPRODUCIBILITY & HARDWARE CONFIG
# ============================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print(f"[HARDWARE] Using GPU: {gpus[0].name}")
    except (RuntimeError, ValueError) as e:
        # GPU context is already initialized (normal in Jupyter/Kaggle environments)
        print(f"[HARDWARE] GPU already initialized. Bypassing config. (Warning: {e})")
else:
    print("[HARDWARE] Running on CPU — sparse attention active.")

# ============================================================
# 1. DATA STRUCTURES & LOADING
# ============================================================
@dataclass
class SubjectData:
    subject_id: str
    class_label: int        
    epochs: np.ndarray      # (N, 9, 3840) float32 [128Hz]
    stage_labels: np.ndarray 
    times: np.ndarray       
    n_epochs: int = field(init=False)

    def __post_init__(self):
        self.n_epochs = self.epochs.shape[0]

def load_all_subjects(data_dir: str) -> List[SubjectData]:
    subjects = []
    seen_ids = set()

    for fname in sorted(os.listdir(data_dir)):
        if not fname.endswith('_data.npy'):
            continue

        subject_id = fname.replace('_data.npy', '')
        if subject_id in seen_ids:
            raise ValueError(f"DUPLICATE: {subject_id} — data leakage risk.")
        seen_ids.add(subject_id)

        class_label = 0 if subject_id.startswith('Normal') else 1
        
        # Load and Decimate to 128Hz (OOM Prevention & Nyquist Safe)
        data = np.load(os.path.join(data_dir, f"{subject_id}_data.npy")).astype(np.float32)
        data = data[:, :, ::2] # 256Hz -> 128Hz (7680 -> 3840 samples)
        
        labels = np.load(os.path.join(data_dir, f"{subject_id}_labels.npy"))
        times  = np.load(os.path.join(data_dir, f"{subject_id}_times.npy"))

        subjects.append(SubjectData(
            subject_id=subject_id,
            class_label=class_label,
            epochs=data,
            stage_labels=labels.astype(np.int32),
            times=times.astype(np.float64)
        ))

    print(f"[DATA] Loaded {len(subjects)} subjects (Decimated to 128Hz).")
    return subjects

def normalize_subject_epochs(epochs: np.ndarray) -> np.ndarray:
    n_ep, n_ch, n_samp = epochs.shape
    flat  = epochs.transpose(1, 0, 2).reshape(n_ch, -1)   
    mu    = flat.mean(axis=1, keepdims=True)
    sigma = flat.std(axis=1, keepdims=True)
    sigma = np.where(sigma < 1e-8, 1.0, sigma)
    normed = ((flat - mu) / sigma).reshape(n_ch, n_ep, n_samp).transpose(1, 0, 2)
    return normed.astype(np.float32)

# ============================================================
# 2. GRAPH CONSTRUCTION (LEAKAGE ERADICATED)
# ============================================================
def build_night_graph(
    times: np.ndarray,
    k_temporal: int = 5,
    tau: float = 120.0
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Construct sparse edge list with Exponential Time-Decay weights.
    add_transition_edges has been completely deleted to prevent data leakage.
    """
    n = len(times)
    edge_set: dict = {}  

    # Type 1: Temporal k-NN with Exponential Decay
    for i in range(n):
        start = max(0, i - k_temporal)
        end = min(n, i + k_temporal + 1)
        
        for j in range(start, end):
            if i == j: continue
            
            delta_t = abs(times[i] - times[j])
            weight = np.exp(-max(0, delta_t - 30.0) / tau)
            
            if weight > 0.01:
                key = (min(i, j), max(i, j))
                edge_set[key] = max(edge_set.get(key, 0.0), weight)

    if not edge_set:
        return np.array([[0, 0]], dtype=np.int32), np.array([1.0], dtype=np.float32)

    edge_keys = np.array(list(edge_set.keys()), dtype=np.int32)       
    edge_weights = np.array(list(edge_set.values()), dtype=np.float32) 

    edge_index = np.concatenate([edge_keys, edge_keys[:, ::-1]], axis=0)
    edge_weight = np.concatenate([edge_weights, edge_weights], axis=0)

    return edge_index, edge_weight

# ============================================================
# 3. STAGE 1: EPOCH ENCODER (128Hz Adapted)
# ============================================================
class MultiResolutionSTFT(tf.keras.layers.Layer):
    def __init__(self, target_time_steps: int = 30, **kwargs):
        super().__init__(**kwargs)
        # Optimized for 128Hz sampling rate
        self.stft_params = [
            (128,  64,  128),  # 1s window
            (256,  128, 256),  # 2s window
            (512,  256, 512),  # 4s window
        ]
        self.target_t = target_time_steps 

    def call(self, x):
        aligned = []
        for frame_len, frame_step, fft_len in self.stft_params:
            stft_out = tf.signal.stft(
                x, frame_length=frame_len,
                frame_step=frame_step, fft_length=fft_len
            )                                   
            mag = tf.math.log1p(tf.abs(stft_out))  
            
            shape  = tf.shape(mag)
            b, ch, t, f = shape[0], shape[1], shape[2], shape[3]
            
            s_flat = tf.reshape(mag, [-1, t, f, 1])
            s_flat = tf.image.resize(s_flat, [self.target_t, f], method='bilinear')
            s_flat = tf.reshape(s_flat, [b, ch, self.target_t, f])
            aligned.append(s_flat)

        return tf.concat(aligned, axis=-1)

class ChannelAttention(tf.keras.layers.Layer):
    def __init__(self, n_channels: int = 9, reduction: int = 3, **kwargs):
        super().__init__(**kwargs)
        self.gap = tf.keras.layers.GlobalAveragePooling2D(data_format='channels_last')
        self.fc1 = tf.keras.layers.Dense(max(1, n_channels // reduction), activation='relu')
        self.fc2 = tf.keras.layers.Dense(n_channels, activation='sigmoid')

    def call(self, x):
        x_perm = tf.transpose(x, [0, 2, 3, 1])
        gap    = self.gap(x_perm)               
        attn   = self.fc2(self.fc1(gap))        
        attn   = tf.reshape(attn, [-1, tf.shape(x)[1], 1, 1])  
        return x * attn

class EpochEncoder(tf.keras.Model):
    def __init__(self, embed_dim: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.stft    = MultiResolutionSTFT(target_time_steps=30)
        self.ch_attn = ChannelAttention(n_channels=9, reduction=3)

        # Scaled down Conv2D to fit in T4 GPU VRAM
        self.conv_block = tf.keras.Sequential([
            tf.keras.layers.Conv2D(16,  (3, 3), padding='same', activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPool2D((2, 2)),
            tf.keras.layers.Conv2D(32,  (3, 3), padding='same', activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPool2D((2, 2)),
            tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.GlobalAveragePooling2D(),
        ])

        self.proj = tf.keras.layers.Dense(embed_dim, activation=None)
        self.ln   = tf.keras.layers.LayerNormalization()

    def call(self, x, training=False):
        spec = self.stft(x)                          
        spec = self.ch_attn(spec)                    
        spec = tf.transpose(spec, [0, 2, 3, 1])      
        h    = self.conv_block(spec, training=training)  
        return self.ln(self.proj(h))                 

# ============================================================
# 4. STAGE 2: SPARSE GATv2 LAYER & READOUT
# ============================================================
class SparseGATv2Layer(tf.keras.layers.Layer):
    def __init__(self, out_dim: int, n_heads: int = 4, dropout_rate: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.n_heads    = n_heads
        self.head_dim   = out_dim // n_heads
        self.out_dim    = out_dim
        self.dropout_rate = dropout_rate

        self.W_l      = None
        self.W_r      = None
        self.attn_vec = None
        self.dropout  = tf.keras.layers.Dropout(dropout_rate)
        self.ln       = tf.keras.layers.LayerNormalization()
        self.leaky    = tf.keras.layers.LeakyReLU(negative_slope=0.2)

    def build(self, input_shape):
        d_in = input_shape[-1] if isinstance(input_shape, tuple) else input_shape
        self.W_l = tf.keras.layers.Dense(self.out_dim, use_bias=False)
        self.W_r = tf.keras.layers.Dense(self.out_dim, use_bias=False)
        self.attn_vec = self.add_weight(
            name='attn_vec', shape=(self.n_heads, self.head_dim),
            initializer='glorot_uniform', trainable=True
        )
        super().build(input_shape)

    def call(self, node_features, edge_index, edge_weight, training=False):
        N = tf.shape(node_features)[0]
        E = tf.shape(edge_index)[0]

        h_l = self.W_l(node_features)  
        h_r = self.W_r(node_features)  

        src_idx = edge_index[:, 0]     
        dst_idx = edge_index[:, 1]     

        h_src = tf.gather(h_l, src_idx)  
        h_dst = tf.gather(h_r, dst_idx)  

        pairwise = self.leaky(h_src + h_dst)  
        pairwise_heads = tf.reshape(pairwise, [E, self.n_heads, self.head_dim])
        
        attn_scores = tf.reduce_sum(
            pairwise_heads * self.attn_vec[tf.newaxis, :, :], axis=-1
        )  
        attn_scores = attn_scores * tf.reshape(edge_weight, [-1, 1])  

        attn_weights = self._sparse_softmax(attn_scores, dst_idx, N)  
        attn_weights = self.dropout(attn_weights, training=training)

        h_r_heads  = tf.reshape(h_r, [N, self.n_heads, self.head_dim])
        src_heads  = tf.gather(h_r_heads, src_idx)  

        weighted   = src_heads * tf.reshape(attn_weights, [E, self.n_heads, 1])

        out = tf.zeros([N, self.n_heads, self.head_dim], dtype=tf.float32)
        out = tf.tensor_scatter_nd_add(
            out,
            tf.reshape(dst_idx, [-1, 1]),
            weighted
        )
        out = tf.reshape(out, [N, self.out_dim])  

        if node_features.shape[-1] == self.out_dim:
            out = self.ln(out + node_features)
        else:
            out = self.ln(out)

        return out, attn_weights

    @staticmethod
    def _sparse_softmax(scores: tf.Tensor, dst_idx: tf.Tensor, n_nodes: int) -> tf.Tensor:
        node_max = tf.math.unsorted_segment_max(scores, dst_idx, num_segments=n_nodes) 
        max_per_edge = tf.gather(node_max, dst_idx)    
        exp_scores = tf.exp(scores - max_per_edge)     
        node_sum = tf.math.unsorted_segment_sum(exp_scores, dst_idx, num_segments=n_nodes) 
        sum_per_edge = tf.gather(node_sum, dst_idx)    
        return exp_scores / (sum_per_edge + 1e-8)      

class HierarchicalReadout(tf.keras.layers.Layer):
    def __init__(self, hidden_dim: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.attn_fc = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden_dim, activation='tanh'),
            tf.keras.layers.Dense(1, activation=None)
        ])

    def call(self, node_features):
        attn_scores  = self.attn_fc(node_features)          
        attn_weights = tf.nn.softmax(attn_scores, axis=0)   
        graph_embed  = tf.reduce_sum(node_features * attn_weights, axis=0)  
        return graph_embed, tf.squeeze(attn_weights, axis=-1)  

# ============================================================
# 5. FULL SG-TAN MODEL (LEAKAGE FIX APPLIED)
# ============================================================
class SGTAN(tf.keras.Model):
    def __init__(
        self,
        epoch_embed_dim: int = 64,
        gat_dim: int = 64,
        gat_heads: int = 4,
        gat_layers: int = 3,
        gat_dropout: float = 0.1,
        clf_dropout: float = 0.3,
        encoder_batch_size: int = 16,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.epoch_encoder     = EpochEncoder(embed_dim=epoch_embed_dim)
        # DELETED: self.stage_embed (No Ground-Truth Leakage allowed)
        
        self.input_proj        = tf.keras.layers.Dense(gat_dim, activation='relu')
        self.encoder_batch_sz  = encoder_batch_size

        self.gat_layers_list = [
            SparseGATv2Layer(out_dim=gat_dim, n_heads=gat_heads, dropout_rate=gat_dropout)
            for _ in range(gat_layers)
        ]

        self.readout = HierarchicalReadout(hidden_dim=32)

        self.classifier = tf.keras.Sequential([
            tf.keras.layers.Dense(32, activation='relu'),
            tf.keras.layers.Dropout(clf_dropout),
            tf.keras.layers.Dense(16, activation='relu'),
            tf.keras.layers.Dropout(clf_dropout),
            tf.keras.layers.Dense(1, activation='sigmoid')
        ])

    def encode_epochs_batched(self, epochs, training: bool):
        n_total = epochs.shape[0]
        bs      = self.encoder_batch_sz
        parts   = []

        for start in range(0, n_total, bs):
            end     = min(start + bs, n_total)
            z_batch = self.epoch_encoder(epochs[start:end], training=training)
            parts.append(z_batch)

        return tf.concat(parts, axis=0)  

    def call(self, epochs, edge_index, edge_weight, training=False):
        # 1. strictly encode raw EEG
        z = self.encode_epochs_batched(epochs, training)  # (N, embed_dim)
        
        # 2. Graph Message Passing
        h = self.input_proj(z)                            # (N, gat_dim)
        gat_attn_list = []
        for gat_layer in self.gat_layers_list:
            h, attn = gat_layer(h, edge_index, edge_weight, training=training)
            gat_attn_list.append(attn)

        # 3. Readout & Classification
        graph_embed, readout_attn = self.readout(h)       
        logit = self.classifier(tf.expand_dims(graph_embed, 0), training=training) 
        prob  = tf.reshape(logit, (1,))  # Prevent 0-d tensor crash

        return prob, readout_attn, gat_attn_list

# ============================================================
# 6. INPUT PREPARATION & TRAINING
# ============================================================
def prepare_subject_inputs(subject: SubjectData):
    normed = normalize_subject_epochs(subject.epochs)
    # Stage labels omitted to ensure 0 leakage
    edge_index, edge_weight = build_night_graph(
        subject.times, k_temporal=5
    )
    
    return (
        tf.constant(normed,         dtype=tf.float32),
        tf.constant(edge_index,     dtype=tf.int32),
        tf.constant(edge_weight,    dtype=tf.float32),
        tf.constant([float(subject.class_label)], dtype=tf.float32), 
    )

def run_loso_cv(subjects: List[SubjectData], n_epochs_train: int = 20, learning_rate: float = 1e-4):
    n_subjects  = len(subjects)
    all_preds   = []
    all_labels  = []
    all_sids    = []

    bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)

    for fold_idx in range(n_subjects):
        test_subject  = subjects[fold_idx]
        train_subjects = [s for i, s in enumerate(subjects) if i != fold_idx]

        print(f"\n{'='*68}\nFOLD {fold_idx+1:2d}/{n_subjects} — Test: {test_subject.subject_id}\n{'='*68}")

        model     = SGTAN(epoch_embed_dim=64, gat_dim=64, encoder_batch_size=16)
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

        train_inputs = [prepare_subject_inputs(s) for s in train_subjects]

        for epoch in range(n_epochs_train):
            epoch_losses = []
            indices      = np.random.permutation(len(train_subjects))

            for idx in indices:
                ep_tf, ei_tf, ew_tf, lb_tf = train_inputs[idx]

                with tf.GradientTape() as tape:
                    prob, _, _ = model(ep_tf, ei_tf, ew_tf, training=True)
                    loss = bce(lb_tf, prob)
                    
                    l2 = tf.add_n([tf.nn.l2_loss(v) for v in model.trainable_variables if 'bias' not in v.name]) * 1e-4
                    total_loss = loss + l2

                grads, _ = tf.clip_by_global_norm(tape.gradient(total_loss, model.trainable_variables), 1.0)
                optimizer.apply_gradients(zip(grads, model.trainable_variables))
                epoch_losses.append(float(loss))

            print(f"  Epoch {epoch+1:2d}/{n_epochs_train} | Loss: {np.mean(epoch_losses):.4f}")

        # ---- TEST ----
        ep_tf, ei_tf, ew_tf, lb_tf = prepare_subject_inputs(test_subject)
        prob_tf, _, _ = model(ep_tf, ei_tf, ew_tf, training=False)

        pred_prob  = float(prob_tf[0])
        pred_label = 1 if pred_prob >= 0.5 else 0
        true_label = test_subject.class_label

        all_preds.append(pred_prob)
        all_labels.append(true_label)
        all_sids.append(test_subject.subject_id)

        status = "✓" if pred_label == true_label else "✗"
        print(f"  [{status}] P(Insomnia)={pred_prob:.4f} | True={true_label}")

        del model, optimizer, train_inputs
        tf.keras.backend.clear_session()
        gc.collect()

    return compute_metrics(all_preds, all_labels, all_sids)

# ============================================================
# 7. METRICS & BASELINE (128Hz Adapted)
# ============================================================
def compute_metrics(preds: List[float], labels: List[int], subject_ids: List[str]) -> Dict:
    pa  = np.array(preds)
    la  = np.array(labels)
    pb  = (pa >= 0.5).astype(int)

    tp = int(np.sum((pb == 1) & (la == 1)))
    tn = int(np.sum((pb == 0) & (la == 0)))
    fp = int(np.sum((pb == 1) & (la == 0)))
    fn = int(np.sum((pb == 0) & (la == 1)))
    N  = tp + tn + fp + fn

    acc  = (tp + tn) / N
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1   = 2 * tp / (2 * tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    print(f"\n{'='*68}\nLOSO RESULTS (N={N})\n{'='*68}")
    print(f"  Accuracy:    {acc:.4f}\n  Sensitivity: {sens:.4f}\n  Specificity: {spec:.4f}\n  F1-Score:    {f1:.4f}")
    return dict(accuracy=acc, sensitivity=sens, specificity=spec, f1=f1, predictions=preds, labels=labels)

def run_baseline_svm(subjects: List[SubjectData]) -> Dict:
    from sklearn.svm import SVC
    from sklearn.preprocessing import StandardScaler

    print(f"\n{'='*68}\nBASELINE: Handcrafted Features + RBF-SVM (LOSO)\n{'='*68}")
    
    # Corrected for 128Hz Nyquist limit
    sfreq = 128.0
    freqs = np.fft.rfftfreq(3840, 1.0 / sfreq)  # 3840 samples
    bands = {
        'delta': (freqs >= 0.5) & (freqs < 4.0),
        'theta': (freqs >= 4.0) & (freqs < 8.0),
        'alpha': (freqs >= 8.0) & (freqs < 13.0),
        'beta':  (freqs >= 13.0) & (freqs < 30.0),
    }

    def extract_subject_features(subject: SubjectData) -> np.ndarray:
        features = []
        fft_mag = np.abs(np.fft.rfft(subject.epochs, axis=-1)) # (N, 9, 1921)
        for ch_idx in range(9):
            mean_spectrum = fft_mag[:, ch_idx, :].mean(axis=0)
            d = mean_spectrum[bands['delta']].mean()
            t = mean_spectrum[bands['theta']].mean()
            a = mean_spectrum[bands['alpha']].mean()
            b = mean_spectrum[bands['beta']].mean()
            features.extend([d, t, a, b, t / (a + 1e-8), d / (b + 1e-8)])
            
        n_ep = subject.n_epochs
        for stage in range(5):
            features.append(float(np.sum(subject.stage_labels == stage)) / n_ep)
        features.append(float(np.sum(subject.stage_labels[:-1] != subject.stage_labels[1:])) / n_ep)
        features.append(float(np.sum(subject.stage_labels != 0)) / n_ep)
        return np.array(features, dtype=np.float32)

    X = np.array([extract_subject_features(s) for s in subjects])  
    y = np.array([s.class_label for s in subjects])                 

    preds, lbls, sids = [], [], []
    for i in range(len(subjects)):
        X_train = np.delete(X, i, axis=0)
        y_train = np.delete(y, i)
        X_test  = X[i:i+1]

        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=SEED)
        svm.fit(X_train, y_train)
        preds.append(float(svm.predict_proba(X_test)[0, 1]))
        lbls.append(int(y[i]))
        sids.append(subjects[i].subject_id)

    return compute_metrics(preds, lbls, sids)

# ============================================================
# MAIN EXECUTION
# ============================================================
if __name__ == "__main__":
    DATA_DIR = "/kaggle/input/datasets/johanliebert28/insominia-sliced-chronological-dataset"
    
    subjects = load_all_subjects(DATA_DIR)
    baseline_results = run_baseline_svm(subjects)
    sgtan_results = run_loso_cv(subjects, n_epochs_train=20)

[HARDWARE] GPU already initialized. Bypassing config. (Warning: Cannot set memory growth on device when virtual devices configured)
[DATA] Loaded 22 subjects (Decimated to 128Hz).

BASELINE: Handcrafted Features + RBF-SVM (LOSO)

LOSO RESULTS (N=22)
  Accuracy:    0.3182
  Sensitivity: 0.3636
  Specificity: 0.2727
  F1-Score:    0.3478

FOLD  1/22 — Test: Insomnia_01
  Epoch  1/20 | Loss: 0.7711
  Epoch  2/20 | Loss: 0.6938
  Epoch  3/20 | Loss: 0.7250
  Epoch  4/20 | Loss: 0.6926
  Epoch  5/20 | Loss: 0.6584
  Epoch  6/20 | Loss: 0.9380
  Epoch  7/20 | Loss: 1.0542
  Epoch  8/20 | Loss: 0.9078
  Epoch  9/20 | Loss: 0.7486
  Epoch 10/20 | Loss: 0.8770
  Epoch 11/20 | Loss: 0.8093
  Epoch 12/20 | Loss: 0.7213
  Epoch 13/20 | Loss: 0.7548
  Epoch 14/20 | Loss: 0.9397
  Epoch 15/20 | Loss: 0.9646
  Epoch 16/20 | Loss: 0.8056
  Epoch 17/20 | Loss: 0.8345
  Epoch 18/20 | Loss: 0.6752
  Epoch 19/20 | Loss: 1.0502
  Epoch 20/20 | Loss: 0.9741
  [✗] P(Insomnia)=0.4674 | True=1

FOLD  2/22 — Te